# 04 — Advanced Modeling (and a Leakage Investigation)

An earlier version of this notebook reported precision, recall, and F1 all
above 0.99 for essentially every model type and every imbalance-handling
strategy. Numbers that clean are not a result to celebrate in a fraud
problem this imbalanced -- they're a signal to stop and check for leakage.
Real-world fraud detection systems typically report performance well below
that (commonly cited in the 0.80-0.95 range on ranking metrics like
PR-AUC/ROC-AUC, often lower on precision at a fixed threshold), so this
notebook treats the near-perfect score as a bug report against itself and
works through it systematically before trusting any result:

1. **Is the train/test split happening before resampling?** (A classic bug:
   fitting SMOTE on the full dataset before splitting lets synthetic
   minority samples leak across train and test.)
2. **Is the time-based split actually a time-based split?** (i.e. not
   accidentally random or overlapping.)
3. **Is a feature itself leaking the label?** (The subtler, and as it turns
   out, real problem here.)

Only after resolving all three do we build the model comparison we actually
trust -- run twice, side by side, so the cost of the leakage is visible
rather than buried in a footnote.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
import joblib
import os

df = pd.read_csv("data/processed/engineered_transactions.csv")

# Time-based split -- identical logic to 03_baseline_modeling.ipynb
cutoff_step = np.quantile(df["step"], 0.8)
train_mask = df["step"] <= cutoff_step
test_mask = ~train_mask

full_feature_cols = [c for c in df.columns if c not in ("step", "nameOrig", "nameDest", "isFraud")]

X_train_full = df.loc[train_mask, full_feature_cols]
y_train = df.loc[train_mask, "isFraud"]
X_test_full = df.loc[test_mask, full_feature_cols]
y_test = df.loc[test_mask, "isFraud"]
amounts_test = df.loc[test_mask, "amount"].values

print(f"Train: {X_train_full.shape}, fraud={y_train.sum()}")
print(f"Test:  {X_test_full.shape}, fraud={y_test.sum()}")


def evaluate(y_true, y_pred, y_score, label):
    return {
        "model": label,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_score),
    }


Train: (2217905, 16), fraud=3955
Test:  (552504, 16), fraud=4258


## Diagnostic 1 & 2: Is the split itself the problem?

Two classic bugs to rule out before looking anywhere else:

- **Resampling before splitting.** If SMOTE (or any resampler) is fit on the
  full dataset and *then* split into train/test, synthetic minority samples
  built from test-set neighbors can end up in training, and vice versa. The
  fix is mechanical: split first, resample only the training fold, and never
  touch the test fold.
- **A time-based split that isn't actually time-ordered** -- e.g. a random
  shuffle that happens to correlate with `step`, or an off-by-one that lets
  a few rows overlap between train and test.

We check both directly rather than assuming the code is correct.


In [2]:
# Diagnostic 1: rows shared between train and test masks (should be none --
# every row belongs to exactly one side of the split)
shared_rows = (train_mask & test_mask).sum()
print(f"Rows in both train AND test: {shared_rows}")

# Diagnostic 2: strict time ordering -- every training step must be earlier
# than every test step, with no overlap
train_max_step = df.loc[train_mask, "step"].max()
test_min_step = df.loc[test_mask, "step"].min()
print(f"Train step range: {df.loc[train_mask, 'step'].min()} - {train_max_step}")
print(f"Test step range:  {test_min_step} - {df.loc[test_mask, 'step'].max()}")
assert train_max_step < test_min_step, "Time split is broken -- train/test overlap in step!"
print("Confirmed: train and test do not overlap in time.")


Rows in both train AND test: 0
Train step range: 1 - 354
Test step range:  355 - 743
Confirmed: train and test do not overlap in time.


Both check out clean:

- `shared_rows` is 0 -- every transaction belongs to exactly one side of the split.
- Every training step (up to 354) is strictly before every test step (355
  onward) -- confirmed by the assertion, not just eyeballed.

And looking at the modeling code itself: `X_train_full`/`y_train` and
`X_test_full`/`y_test` are created from `train_mask`/`test_mask` **before**
any resampler is touched, and later in this notebook, `SMOTE.fit_resample()`
and `RandomUnderSampler.fit_resample()` are only ever called on the training
arrays -- the test set is held out, untouched, and evaluated on exactly
once per model. So: **not a split bug, and not a resampling-order bug.**
That means the near-perfect scores are coming from somewhere else --
the features themselves.


## Diagnostic 3: Feature-level leakage

To find out which features are doing the work, we train one exploratory
Random Forest on the full feature set and inspect its feature importances
directly, before doing anything else.


In [3]:
diagnostic_rf = RandomForestClassifier(
    n_estimators=100, max_depth=12, class_weight="balanced", n_jobs=-1, random_state=42,
)
diagnostic_rf.fit(X_train_full, y_train)
diagnostic_score = diagnostic_rf.predict_proba(X_test_full)[:, 1]
diagnostic_pred = (diagnostic_score >= 0.5).astype(int)

print(pd.DataFrame([evaluate(y_test, diagnostic_pred, diagnostic_score, "Diagnostic RF (full features)")])
      .set_index("model").round(4))

importances = pd.Series(diagnostic_rf.feature_importances_, index=full_feature_cols) \
    .sort_values(ascending=False)
print("\nTop 10 features by importance:")
importances.head(10)


                               precision  recall      f1  pr_auc
model                                                           
Diagnostic RF (full features)     0.9998  0.9988  0.9993     1.0

Top 10 features by importance:


errorBalanceOrig        0.426091
oldbalanceOrg           0.178629
newbalanceOrig          0.102370
newbalanceDest          0.090898
amount                  0.047156
errorBalanceDest        0.044935
hour_of_day             0.035836
oldbalanceDest          0.030649
dest_txn_count_prior    0.015076
type_TRANSFER           0.012809
dtype: float64

There it is. `errorBalanceOrig` alone accounts for well over a third of this
model's total feature importance, and together with `oldbalanceOrg`,
`newbalanceOrig`, `newbalanceDest`, and `errorBalanceDest`, the
post-transaction-derived features account for the large majority of what
the model is using to make decisions.

That's the leakage. Recall from [02_feature_engineering.ipynb](02_feature_engineering.ipynb):

```
errorBalanceOrig = newbalanceOrig + amount - oldbalanceOrg
errorBalanceDest = oldbalanceDest + amount - newbalanceDest
```

Both are built directly from `newbalanceOrig` / `newbalanceDest` --
fields that only exist *after* a transaction has completed. A real-time
authorization system does not have these numbers when it needs to make a
decision. Worse, PaySim's fraud simulator has a mechanical quirk: it almost
always drains the origin account to exactly zero when generating a
fraudulent transaction, regardless of the stated amount. We can check this
directly:


In [4]:
fraud_rows = df[df["isFraud"] == 1]
print("Share of fraud where newbalanceOrig == 0 (account fully drained):",
      (fraud_rows["newbalanceOrig"] == 0).mean())
print("Share of fraud where oldbalanceOrg == amount (drained exactly):",
      ((fraud_rows["oldbalanceOrg"] - fraud_rows["amount"]).abs() < 1).mean())


Share of fraud where newbalanceOrig == 0 (account fully drained): 0.9805186898818946
Share of fraud where oldbalanceOrg == amount (drained exactly): 0.9782052843053696


Roughly 98% of fraud cases in this dataset show that exact signature. It's
a real pattern in the data, but it's a **simulation artifact of how PaySim
generates fraud**, not a generalizable insight about how real fraudsters
behave, and it's only visible *after* the money has already moved. A model
that leans on it is measuring "can I detect PaySim's fraud-generation
logic," not "can I detect fraud before it happens."

**The fix:** treat `newbalanceOrig`, `newbalanceDest`, `errorBalanceOrig`,
and `errorBalanceDest` as off-limits for any model claiming to be
deployable, and rebuild the entire comparison without them -- not as a
side note, but as the primary result.


In [5]:
excluded_post_transaction_fields = [
    "newbalanceOrig", "newbalanceDest", "errorBalanceOrig", "errorBalanceDest",
]
pretxn_feature_cols = [c for c in full_feature_cols if c not in excluded_post_transaction_fields]

print(f"Full feature set ({len(full_feature_cols)}):", full_feature_cols)
print(f"\nPre-transaction feature set ({len(pretxn_feature_cols)}):", pretxn_feature_cols)


Full feature set (16): ['type_TRANSFER', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'hour_of_day', 'day', 'orig_txn_count_prior', 'orig_time_since_last_txn', 'orig_dual_role_prior', 'dest_txn_count_prior', 'dest_time_since_last_txn', 'dest_dual_role_prior']

Pre-transaction feature set (12): ['type_TRANSFER', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'hour_of_day', 'day', 'orig_txn_count_prior', 'orig_time_since_last_txn', 'orig_dual_role_prior', 'dest_txn_count_prior', 'dest_time_since_last_txn', 'dest_dual_role_prior']


## Corrected comparison: two tracks, run side by side

We rerun the same 6-way comparison (Random Forest and XGBoost, each with
class weighting / SMOTE / random undersampling) **twice**:

- **Full-feature** -- includes the post-transaction fields. Reported only as
  an upper-bound / unrealistic benchmark, since it can't be deployed as a
  real-time authorization check.
- **Pre-transaction** -- restricted to fields known before a transaction
  completes. This is the track we treat as the project's real result going
  forward.

As before, all resampling is fit on the training fold only, per feature set
(SMOTE's synthetic points depend on which features it computes nearest
neighbors in, so it has to be refit for the reduced feature set rather than
reused).


In [6]:
def build_resampled_datasets(X_train, y_train):
    datasets = {"class_weight": (X_train, y_train)}

    smote = SMOTE(sampling_strategy=0.1, random_state=42)
    X_smote, y_smote = smote.fit_resample(X_train, y_train)
    datasets["smote"] = (X_smote, y_smote)

    rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
    X_under, y_under = rus.fit_resample(X_train, y_train)
    datasets["undersample"] = (X_under, y_under)

    return datasets


def run_comparison(datasets, X_test, y_test, scale_pos_weight):
    results = []
    fitted_models = {}
    for strategy, (X_s, y_s) in datasets.items():
        rf_class_weight = "balanced" if strategy == "class_weight" else None
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=12, class_weight=rf_class_weight,
            n_jobs=-1, random_state=42,
        )
        rf.fit(X_s, y_s)
        rf_score = rf.predict_proba(X_test)[:, 1]
        rf_pred = (rf_score >= 0.5).astype(int)
        results.append(evaluate(y_test, rf_pred, rf_score, f"Random Forest + {strategy}"))
        fitted_models[f"rf_{strategy}"] = rf

        xgb_scale_pos_weight = scale_pos_weight if strategy == "class_weight" else 1.0
        xgbm = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, tree_method="hist",
            scale_pos_weight=xgb_scale_pos_weight, eval_metric="aucpr", random_state=42,
        )
        xgbm.fit(X_s, y_s)
        xgb_score = xgbm.predict_proba(X_test)[:, 1]
        xgb_pred = (xgb_score >= 0.5).astype(int)
        results.append(evaluate(y_test, xgb_pred, xgb_score, f"XGBoost + {strategy}"))
        fitted_models[f"xgb_{strategy}"] = xgbm

    return pd.DataFrame(results).set_index("model").round(4), fitted_models


scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Track 1: full-feature (unrealistic upper bound)
full_datasets = build_resampled_datasets(X_train_full, y_train)
full_comparison, full_models = run_comparison(full_datasets, X_test_full, y_test, scale_pos_weight)

# Track 2: pre-transaction only (deployable / realistic)
X_train_pre = X_train_full[pretxn_feature_cols]
X_test_pre = X_test_full[pretxn_feature_cols]
pretxn_datasets = build_resampled_datasets(X_train_pre, y_train)
pretxn_comparison, pretxn_models = run_comparison(pretxn_datasets, X_test_pre, y_test, scale_pos_weight)


In [7]:
print("=== FULL-FEATURE (upper bound / unrealistic) ===")
full_comparison


=== FULL-FEATURE (upper bound / unrealistic) ===


,precision,recall,f1,pr_auc
model,,,,
Random Forest + class_weight,0.9998,0.9988,0.9993,1.0000
XGBoost + class_weight,0.9991,0.9995,0.9993,0.9998
Random Forest + smote,0.9991,0.9995,0.9993,1.0000
XGBoost + smote,0.9986,0.9984,0.9985,0.9998
Random Forest + undersample,0.9970,0.9998,0.9984,0.9998
XGBoost + undersample,0.9937,0.9986,0.9961,0.9992


In [8]:
print("=== PRE-TRANSACTION ONLY (deployable / realistic) ===")
pretxn_comparison


=== PRE-TRANSACTION ONLY (deployable / realistic) ===


,precision,recall,f1,pr_auc
model,,,,
Random Forest + class_weight,0.3512,0.8713,0.5006,0.8204
XGBoost + class_weight,0.8318,0.8283,0.8301,0.8898
Random Forest + smote,0.7327,0.6980,0.7149,0.7872
XGBoost + smote,0.9450,0.7384,0.8290,0.9073
Random Forest + undersample,0.5630,0.8372,0.6733,0.8305
XGBoost + undersample,0.6497,0.8967,0.7534,0.9114


In [9]:
# Side-by-side view of the same model/strategy row under both feature sets --
# this is the performance gap that was hidden in the original, single-track version.
side_by_side = pd.concat(
    {
        "Full-feature (upper bound)": full_comparison[["precision", "recall", "f1", "pr_auc"]],
        "Pre-transaction (deployable)": pretxn_comparison[["precision", "recall", "f1", "pr_auc"]],
    },
    axis=1,
)
side_by_side


Full-feature (upper bound)                  \
                                              precision  recall      f1   
model                                                                     
Random Forest + class_weight                     0.9998  0.9988  0.9993   
XGBoost + class_weight                           0.9991  0.9995  0.9993   
Random Forest + smote                            0.9991  0.9995  0.9993   
XGBoost + smote                                  0.9986  0.9984  0.9985   
Random Forest + undersample                      0.9970  0.9998  0.9984   
XGBoost + undersample                            0.9937  0.9986  0.9961   

                                     Pre-transaction (deployable)          \
                              pr_auc                    precision  recall   
model                                                                       
Random Forest + class_weight  1.0000                       0.3512  0.8713   
XGBoost + class_weight        0.9998                       0.8318  0.8283   
Random Forest + smote         1.0000                       0.7327  0.6980   
XGBoost + smote               0.9998                       0.9450  0.7384   
Random Forest + undersample   0.9998                       0.5630  0.8372   
XGBoost + undersample         0.9992                       0.6497  0.8967   

                                              
                                  f1  pr_auc  
model                                         
Random Forest + class_weight  0.5006  0.8204  
XGBoost + class_weight        0.8301  0.8898  
Random Forest + smote         0.7149  0.7872  
XGBoost + smote               0.8290  0.9073  
Random Forest + undersample   0.6733  0.8305  
XGBoost + undersample         0.7534  0.9114

### Reading this side by side

Every single row drops substantially from the full-feature column to the
pre-transaction column. That drop **is the size of the leak** -- it's
exactly how much performance was coming from fields that don't exist yet at
decision time, rather than from patterns a production system could actually
use.

The pre-transaction numbers are also just more *believable*: precision/recall
in the 0.3-0.9 range and PR-AUC around 0.8, not 0.999+ across the board.
That's consistent with what's typically reported in the literature for
fraud detection restricted to pre-authorization features, and it's the set
of numbers we treat as this project's real result from here on. The
full-feature column is kept only as a diagnostic upper bound and for the
interpretability side-note in [05_interpretability.ipynb](05_interpretability.ipynb)
-- it is not a claim about deployable performance.


In [10]:
best_full_model_name = full_comparison["pr_auc"].idxmax()
best_pretxn_model_name = pretxn_comparison["pr_auc"].idxmax()


def model_key(name):
    return name.lower().replace(" + ", "_").replace("random forest", "rf").replace("xgboost", "xgb")


best_full_model = full_models[model_key(best_full_model_name)]
best_pretxn_model = pretxn_models[model_key(best_pretxn_model_name)]
best_pretxn_score_test = best_pretxn_model.predict_proba(X_test_pre)[:, 1]

print("Best full-feature model (upper bound, not deployable):", best_full_model_name)
print(full_comparison.loc[[best_full_model_name]])
print()
print("Best pre-transaction model (this project's primary result):", best_pretxn_model_name)
print(pretxn_comparison.loc[[best_pretxn_model_name]])


Best full-feature model (upper bound, not deployable): Random Forest + class_weight
                              precision  recall      f1  pr_auc
model                                                          
Random Forest + class_weight     0.9998  0.9988  0.9993     1.0

Best pre-transaction model (this project's primary result): XGBoost + undersample
                       precision  recall      f1  pr_auc
model                                                   
XGBoost + undersample     0.6497  0.8967  0.7534  0.9114


## Threshold tuning, in business terms -- on the pre-transaction model

Every business decision from here on should be based on the pre-transaction
model, since that's the one that could actually run in production. As in
the original approach, we sweep the decision threshold and report what
happens in terms a fraud operations team would actually act on: what share
of fraud we catch, what share of transaction *volume* gets sent for review,
and what share of fraud *dollars* we recover.


In [11]:
def business_threshold_table(y_true, y_score, amounts, thresholds):
    y_true = np.asarray(y_true)
    total_fraud_dollars = amounts[y_true == 1].sum()
    rows = []
    for t in thresholds:
        preds = (y_score >= t).astype(int)
        tp = (preds == 1) & (y_true == 1)
        n_flagged = preds.sum()
        rows.append({
            "threshold": t,
            "pct_of_transactions_flagged": n_flagged / len(y_true),
            "pct_of_fraud_caught (recall)": tp.sum() / max((y_true == 1).sum(), 1),
            "precision_of_flags": tp.sum() / max(n_flagged, 1),
            "pct_of_fraud_dollars_caught": amounts[tp].sum() / total_fraud_dollars if total_fraud_dollars else np.nan,
        })
    return pd.DataFrame(rows)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
business_table = business_threshold_table(y_test, best_pretxn_score_test, amounts_test, thresholds)
business_table.round(4)


,threshold,pct_of_transactions_flagged,pct_of_fraud_caught (recall),precision_of_flags,pct_of_fraud_dollars_caught
0,0.1,0.0198,0.9629,0.3743,0.9980
1,0.2,0.0155,0.9429,0.4690,0.9961
2,0.3,0.0132,0.9239,0.5374,0.9944
3,0.4,0.0117,0.9096,0.5996,0.9929
4,0.5,0.0106,0.8967,0.6497,0.9913
5,0.6,0.0097,0.8823,0.7004,0.9897
6,0.7,0.0089,0.8699,0.7498,0.9882
7,0.8,0.0081,0.8556,0.8094,0.9864
8,0.9,0.0074,0.8314,0.8687,0.9829


In [12]:
recommended = business_table.iloc[(business_table["pct_of_fraud_caught (recall)"] - 0.80).abs().idxmin()]
print(
    f"Example: at threshold {recommended['threshold']:.1f}, the pre-transaction model catches "
    f"{recommended['pct_of_fraud_caught (recall)']:.1%} of fraud cases "
    f"({recommended['pct_of_fraud_dollars_caught']:.1%} of fraud dollars), "
    f"while sending {recommended['pct_of_transactions_flagged']:.2%} of all "
    f"transactions for review."
)


Example: at threshold 0.9, the pre-transaction model catches 83.1% of fraud cases (98.3% of fraud dollars), while sending 0.74% of all transactions for review.


This is the number that should inform an actual authorization policy: not
the full-feature model's inflated 99.9%, but this more modest,
believable trade-off curve, achieved using only information available
before the transaction completes.

## Saving artifacts

We save **both** models -- the pre-transaction model as the project's
primary result, and the full-feature model strictly as a labeled upper-bound
reference used in 05 to visually confirm the leakage story via SHAP.


In [13]:
os.makedirs("models", exist_ok=True)
joblib.dump(best_full_model, "models/best_full_feature_model.joblib")
joblib.dump(best_pretxn_model, "models/best_pretransaction_model.joblib")
joblib.dump({
    "full_feature_cols": full_feature_cols,
    "pre_txn_feature_cols": pretxn_feature_cols,
    "best_full_model_name": best_full_model_name,
    "best_pretxn_model_name": best_pretxn_model_name,
    "primary_model": "pretransaction",
}, "models/model_metadata.joblib")

print("Saved models/ artifacts. Primary model for 05/06 is the PRE-TRANSACTION model:")
print(" ", best_pretxn_model_name)


Saved models/ artifacts. Primary model for 05/06 is the PRE-TRANSACTION model:
  XGBoost + undersample


## Summary: what we found, and what's real going forward

- **Split-before-resample:** confirmed correct. Train/test masks are
  created first; SMOTE and undersampling are fit only on the training fold.
  No test-set contamination here.
- **Time-based split:** confirmed correct. Train covers steps 1-354, test
  covers steps 355-743, with an explicit assertion that there's zero
  overlap. Not a random shuffle in disguise.
- **Feature-level leakage: confirmed, and this was the real issue.**
  `errorBalanceOrig`, `errorBalanceDest`, `newbalanceOrig`, and
  `newbalanceDest` are all post-transaction fields (the first two are
  literally computed from the other two), and together they drove the
  overwhelming majority of the full-feature model's decisions -- which is
  exactly why it scored 0.999+ across every strategy. That level of
  performance reflected PaySim's fraud-generation mechanics, not a
  generalizable model of real fraud behavior.

**Going forward, the pre-transaction model is this project's primary
result.** Its more modest, more believable numbers
(precision/recall in a realistic range, PR-AUC around 0.8 rather than 1.0)
are what get carried into [05_interpretability.ipynb](05_interpretability.ipynb)
and [06_error_analysis.ipynb](06_error_analysis.ipynb). The full-feature
model is retained only as a labeled upper-bound / diagnostic artifact --
never again presented as a deployable result.
